# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Initial Setup
First, let's print a simple message to ensure our environment is set up correctly.

In [5]:
print("Hello World")

Hello World


Then, let's add the directory containing src to path

In [6]:
import os
#os.chdir('..')
print("Current Working Directory " , os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Checking System Memory
We will check the available system memory to ensure that we have enough resources to load and run the model. The following command outputs the total, free, and available memory in gigabytes.

In [4]:
#!pip show bitsandbytes
#!pip show accelerate
#!pip show transformers
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

MemTotal: 1007.72 GB
MemFree: 817.11 GB
MemAvailable: 970.11 GB
Free GPU Memory (GB): 39.3936


## 3. Code Formatting and Linting
We use `black` for code formatting and `pylint` for linting to ensure our code is clean and follows best practices.


In [6]:
!black notebooks/Llama-3-8B-quant.ipynb
!pylint notebooks/Llama-3-8B-quant.ipynb

All done! ✨ 🍰 ✨
1 file left unchanged.
************* Module Llama-3-8B-quant
notebooks/Llama-3-8B-quant.ipynb:9:0: C0301: Line too long (182/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:39:0: C0301: Line too long (196/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:61:0: C0301: Line too long (239/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:70:0: C0301: Line too long (123/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:95:0: C0301: Line too long (136/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:96:0: C0301: Line too long (101/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:123:0: C0301: Line too long (167/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:137:0: C0301: Line too long (237/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:170:0: C0301: Line too long (148/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:204:0: C0301: Line too long (159/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:235:0: C0301: Line too long

## 4. Loading Environment Variables
We load the Hugging Face token from an environment variable to authenticate our session. This token is necessary to access the model from the Hugging Face Hub.


In [17]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load environment variables from the .env file
load_dotenv()

# Read the Hugging Face token from the environment variable
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

# Log in using the token
login(token=huggingface_token)

Hugging Face token loaded successfully.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /nfs/homedirs/daro/.cache/huggingface/token
Login successful


## 5. Checking CUDA Availability
We check if CUDA is available on the system. CUDA is essential for running the model on GPU, which significantly speeds up the computations.


In [18]:
import torch

# Check CUDA availability
if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB


## 6.1. AutoModelForCausalLM Generation for TinyLlama

In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda"

# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"  # For instruction-based models.
# model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "microsoft/Phi-3-vision-128k-instruct"  # Small enough to run on a gpu_gtx1080.

model_family, model_identifier = model_name.split("/")

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device)

Generating memory footprint of the model in Gigabytes

In [14]:
memory_footprint = model.get_memory_footprint()
print(f"Model Memory Footprint: {(memory_footprint / (1024 ** 3)):.2f} GB")

Model Memory Footprint: 4.10 GB


Example inference

In [22]:
# input_text = "Once upon a time, a curious fox..."
input_text = "What famous tower is in Paris?"

# Encode input text to tensor and move to device
input_ids = tokenizer(input_text, return_tensors="pt").to(device)

generated_ids = model.generate(
    input_ids=input_ids["input_ids"],
    max_length=512,  # Adjust maximum output length
    num_beams=5,  # Adjust number of beams for beam search
)

# Decode generated IDs back to text
generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Print generated text
print("Generated Text:", generated_text)

Generated Text: What famous tower is in Paris?

Student: The Eiffel Tower.

Teacher: The Eiffel Tower is a famous tower located in Paris, France. It was designed by Gustave Eiffel for the 1889 World's Fair and was completed in 1889. The tower stands at 324 meters (1,063 feet) tall, making it the tallest structure in Paris and one of the tallest in the world.

Student: Wow, the Eiffel Tower is so tall!

Teacher: Yes, it's truly a marvel of engineering and architecture. The tower is made of wrought iron and steel, and it's designed to withstand earthquakes and other natural disasters. It's a popular tourist attraction, and millions of people visit the tower each year.

Student: Wow, I've heard of the Eiffel Tower before, but I didn't know it was so tall.

Teacher: Yes, the Eiffel Tower is one of the most recognizable landmarks in the world. It's a symbol of France and a testament to the ingenuity and creativity of Gustave Eiffel and his team.

Student: It's amazing how the Eiffel Tower h

In [4]:
import json

results = {"input": input_text, "output": generated_text}
with open("results.json", "w") as f:
    json.dump(results, f)

## 7. Quantization

In [20]:
device="cuda"

# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"  # For instruction-based models.
# model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "microsoft/Phi-3-vision-128k-instruct"  # Small enough to run on a gpu_gtx1080.

### 7.1. BitsAndBytes

In [21]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    load_in_4bit=False,
    llm_int8_threshold=6.0,
    # llm_int8_skip_modules=["lm_head"],
    llm_int8_enable_fp32_cpu_offload=False,
    llm_int8_has_fp16_weight=False,
)
bnb_config_4bit = BitsAndBytesConfig(
    load_in_8bit=False,
    load_in_4bit=True,
    # bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4",
    # bnb_4bit_quant_type="nf4"
    bnb_4bit_use_double_quant=False,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model_bnb_8bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_8bit, 
    torch_dtype=torch.float32,
    device_map=device
)
model_bnb_4bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_4bit, 
    torch_dtype=torch.float32,
    device_map=device
)

In [15]:
print(f"8bit {model} Memory Footprint: {(model_bnb_8bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")
print(f"4bit Model Memory Footprint: {(model_bnb_4bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")

8bit Model Memory Footprint: 1.39 GB
4bit Model Memory Footprint: 0.94 GB


In [25]:
import time

def prompt_large_language_model(model, tokenizer, input_text, device=None):
    """
    Prompts a large language model by generating text based on an input string.

    Args:
        model_name (str): Name of the pre-trained model from the Hugging Face Hub.
        input_text (str): Input string for the model to generate text from.
        device (str, optional): Device to use for computations (CPU or GPU). Defaults to "cuda" if available, otherwise "cpu".

    Returns:
        None: Prints the generated text to the console.
    """

    try:
        # Start time measurement
        start_time = time.time()
        
        # Convert input text to tensor and move to device
        inputs = tokenizer(input_text, return_tensors="pt").to(device)

        # Generate text using beam search (modify parameters as needed)
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            max_length=512,  # Adjust maximum output length
            num_beams=5,  # Adjust number of beams for beam search
        )

        # Decode generated IDs back to text
        generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        
        # End time measurement
        elapsed_time = time.time() - start_time

        # Print generated text and elapsed time
        print(f"Generated text for model '{model.config._name_or_path}':\n{generated_text}")
        print(f"Elapsed time for text generation: {elapsed_time:.4f} seconds")

    except Exception as e:
        print(f"An error occurred: {e}")


input_text = "Once upon a time"

prompt_large_language_model(model, tokenizer, input_text, device)
prompt_large_language_model(model_bnb_8bit, tokenizer, input_text, device)
prompt_large_language_model(model_bnb_4bit, tokenizer, input_text, device)

Generated text for model 'TinyLlama/TinyLlama-1.1B-Chat-v1.0':
Once upon a time, in a land far, far away, there lived a beautiful princess named Aurora. Aurora was the daughter of the king and queen, and she was loved by all who knew her. One day, Aurora's kingdom was threatened by an evil sorcerer named Maleficent. Maleficent was a powerful sorceress who had been banished from the land by the king and queen. Maleficent had sworn to destroy Aurora and all who lived in her kingdom, and she would stop at nothing to achieve her goal. Aurora knew that she had to find a way to defeat Maleficent and save her kingdom. So, Aurora set out on a perilous journey to find the magical crystal that was said to be the key to defeating Maleficent. Along the way, Aurora encountered many obstacles and challenges, but she never gave up. She was determined to find the crystal and defeat Maleficent, no matter what it took. One day, Aurora stumbled upon a hidden cave deep in the forest. Inside the cave, Auro

/nfs/students/daro/miniconda3/envs/env-quant-rel/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:316: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Generated text for model 'TinyLlama/TinyLlama-1.1B-Chat-v1.0':
Once upon a time, in a land far, far away, there lived a young girl named Lily. Lily was a kind and gentle soul, always looking for ways to make the world a better place. One day, while wandering through the woods, Lily stumbled upon a mysterious old tree. The tree was covered in strange markings, and Lily couldn't help but feel drawn to it. As she approached the tree, she saw that it was home to a magical creature. The creature was a fairy, and it welcomed Lily with open arms. The fairy explained that the tree was a portal to another world, and that Lily had the power to open it. Lily was skeptical at first, but the fairy insisted that she try. So, Lily took a deep breath and stepped through the portal. When she emerged on the other side, she was amazed by what she saw. The world was unlike anything she had ever seen before. The sky was a deep shade of purple, and the trees were covered in vibrant flowers. The ground was l

## 8. Loading WikiText

In [23]:
# Initialize the datamodule
from src.data.WikiTextDataModule import WikiTextDataModule

directory_dataset = os.getcwd()
batch_size = 64
sequence_length = 2048
seed = 1

data_module = WikiTextDataModule(
  directory_dataset=directory_dataset,
  batch_size=batch_size,
  sequence_length=sequence_length,
  tokenizer_name=model_name,
  seed=seed
)

train_dataloader = data_module.train_dataloader()
print(train_dataloader.__dict__)

Token indices sequence length is longer than the specified maximum sequence length for this model (2824491 > 2048). Running this sequence through the model will result in indexing errors


{'dataset': <src.data.WikiTextDataModule.TextDataset object at 0x7fe4e411fb90>, 'num_workers': 0, 'prefetch_factor': None, 'pin_memory': False, 'pin_memory_device': '', 'timeout': 0, 'worker_init_fn': None, '_DataLoader__multiprocessing_context': None, '_dataset_kind': 0, 'batch_size': 64, 'drop_last': False, 'sampler': <torch.utils.data.sampler.SequentialSampler object at 0x7fe4f407f5f0>, 'batch_sampler': <torch.utils.data.sampler.BatchSampler object at 0x7fe4e66f2960>, 'generator': None, 'collate_fn': <function default_collate at 0x7fe505250ae0>, 'persistent_workers': False, '_DataLoader__initialized': True, '_IterableDataset_len_called': None, '_iterator': None}


In [32]:
dataset_size = len(train_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")

for i, (data, target) in enumerate(train_dataloader):
    if i < 3:  # Print the first 3 elements (adjust as needed)
        print(f"Batch {i+1}:")

        # Get the original text from the first element of data
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)

        # Print a few tokens from the input data (original text)
        print(f"  Original Text (first 5 tokens): {original_text[:500]}")

        # Print a few tokens from the input data (tokenized)
        print(f"  Input data (first 5 tokens): {data[0][:5]}")  # Existing line

        # Print a few tokens from the target labels (tokenized)
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")  # Existing line

        # Print the complete shape of data and target
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")
        break  # Stop after printing 3 elements


Number of batches in train_dataloader: 22
Batch 1:
  Original Text (first 5 tokens):   = Valkyria Chronicles III = 
   Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs
  Input data (first 5 tokens): tensor([   1,  259,  353,  478, 2235])
  Target labels (first 5 tokens): tensor([  259,   353,   478,  2235, 29891])
  Input data shape: torch.Size([64, 2048])
  Target labels shape: torch.Size([64, 2048])


In [ ]:
# Evaluate perplexity on each model
from src.evaluations.evaluate_text_generation import evaluate_perplexity


perplexity_8bit = evaluate_perplexity(model_bnb_8bit, train_dataloader, device)
perplexity_4bit = evaluate_perplexity(model_bnb_4bit, train_dataloader, device)

# Print perplexity results
print(f"Perplexity (8-bit): {perplexity_8bit:.4f}")
print(f"Perplexity (4-bit): {perplexity_4bit:.4f}")

## 9. Text Streamer

In [14]:
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

## 10. Code Dump

In [ ]:
import accelerate
print(accelerate.__file__)

import transformers
print(transformers.__file__)

from transformers import BitsAndBytesConfig
print(BitsAndBytesConfig.__dict__)

#!pip index versions accelerate
#!pip install accelerate --force-reinstall
!pip show transformers
!pip index versions transformers
!pip show accelerate
!pip index versions accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, AwqConfig
from accelerate.utils import load_and_quantize_model
from accelerate import init_empty_weights

device="cuda"

# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"  # For instruction-based models.
# model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "microsoft/Phi-3-vision-128k-instruct"  # Small enough to run on a gpu_gtx1080.

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
with init_empty_weights():
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device)
    
# Setting quantization bits for the model
weight_quantization_bits = 8
double_quant = False

bnb_config = BitsAndBytesConfig(
    load_in_8bit=(weight_quantization_bits == 8),
    load_in_4bit=(weight_quantization_bits == 4),
    llm_int8_threshold=6.0,
    llm_int8_skip_modules=["lm_head"],
    llm_int8_enable_fp32_cpu_offload=False,
    llm_int8_has_fp16_weight=False,
    # bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4",
    # bnb_4bit_quant_type="nf4"
    bnb_4bit_use_double_quant=double_quant,
)

bnb_config.skip_modules = None

smashed_model_bnb = load_and_quantize_model(model, bnb_quantization_config=bnb_config, device_map=device)
print(smashed_model_bnb.__class__.__name__)

# Calibration Dataset needed - WikiText? Something else because data leakage? TODO: Explore
# smashed_model_awq = AutoModelForCausalLM.from_pretrained(
#     temp_dir, quantization_config=awq_config, trust_remote_code=True
# )